# 🔬 Kinetics-700 Human Action Recognition — Compact 3D CNN
**Train · Validate · Analyse · Grad-CAM** | GPU/CPU Auto | Checkpoint Resume

All outputs saved to **Google Drive** at `My Drive/Kinetics700-A/temp/`

In [ ]:
import subprocess, sys, os
try: subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError: print("ℹ️ nvidia-smi not found — CPU runtime")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "opencv-python-headless", "scikit-learn", "seaborn", "tqdm",
    "thop", "imageio[ffmpeg]", "tensorboard", "decord"], check=True)
subprocess.run(["apt-get", "install", "-qq", "graphviz"], check=True)

import torch
HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
print(f"💻 Device: {DEVICE.upper()}", end="")
if HAS_GPU:
    print(f" | GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    torch.set_float32_matmul_precision('high')
    print("⚡ CUDNN benchmark ON | TF32 matmul ON")
else:
    print(" | ⚠️ No GPU — training will be SLOW but functional")

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/Kinetics700-A")
DATASET_DIR = BASE / "videos" / "train"
TEMP        = BASE / "temp"
CKPT_DIR    = TEMP / "checkpoints";   CKPT_DIR.mkdir(parents=True, exist_ok=True)
SAVED_DIR   = TEMP / "saved_model";   SAVED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR     = TEMP / "figures";       FIG_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR = TEMP / "metrics";       METRICS_DIR.mkdir(parents=True, exist_ok=True)
TB_DIR      = TEMP / "tensorboard";   TB_DIR.mkdir(parents=True, exist_ok=True)
GRADCAM_DIR = TEMP / "gradcam";       GRADCAM_DIR.mkdir(parents=True, exist_ok=True)
INDEX_PATH  = TEMP / "file_index.json"

print(f"✓ Dataset: {DATASET_DIR} (exists={DATASET_DIR.exists()})")
print(f"✓ Temp:    {TEMP}")

In [ ]:
EXPERIMENT  = "kinetics700_3dcnn"
EPOCHS      = 30          # With BatchNorm3D, the model converges in 30 epochs instead of 100!
BATCH_SIZE  = 8           # Reduced physical batch size to easily fit within 6GB VRAM GTX 1660
ACCUMULATION_STEPS = 4    # 8 * 4 = 32 virtual batch size for identical gradient updates
N_FRAMES    = 10
FRAME_STEP  = 15
IMG_SIZE    = (224, 224)
LR          = 5e-4        # Higher learning rate is now fully stable with BatchNorm3d (5x speedup!)
ADAM_BETAS  = (0.9, 0.999)
VAL_SPLIT   = 0.20
SEED        = 42
DROPOUT     = 0.3
USE_AMP     = HAS_GPU
NUM_WORKERS = 0           # Set to 0 to bypass Colab's extremely tight /dev/shm shared memory limit (prevents worker SIGKILL crashes)
PREFETCH    = 2
CONV_FILTERS = [32, 64, 128]
MAX_SAMPLES_PER_CLASS = 20  # Cap per class — keeps epoch fast (~7000 total vids)
USE_CACHE    = True           # True = use .npy cache (fast), False = decode raw video
CACHE_WORKERS = os.cpu_count() or 2
CACHE_DIR    = Path("/content/cache")
print(f"⚙️ Batch={BATCH_SIZE} | VirtualBatch={BATCH_SIZE*ACCUMULATION_STEPS} | AMP={USE_AMP} | MaxPerClass={MAX_SAMPLES_PER_CLASS} | Cache={'ON' if USE_CACHE else 'OFF'} | CacheWorkers={CACHE_WORKERS}")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Compact3DCNN(nn.Module):
    def __init__(self, num_classes, dropout=DROPOUT):
        super().__init__()
        # Conv3D -> BatchNorm3D -> ReLU -> MaxPool3D
        self.conv1 = nn.Sequential(
            nn.Conv3d(3, 32, 3, padding=1, bias=False), 
            nn.BatchNorm3d(32),
            nn.ReLU(False), 
            nn.MaxPool3d(2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv3d(32, 64, 3, padding=1, bias=False), 
            nn.BatchNorm3d(64),
            nn.ReLU(False), 
            nn.MaxPool3d(2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv3d(64, 128, 3, padding=1, bias=False), 
            nn.BatchNorm3d(128),
            nn.ReLU(False), 
            nn.MaxPool3d(2)
        )
        self.drop = nn.Dropout(dropout)
        self.gap  = nn.AdaptiveAvgPool3d(1)
        self.fc   = nn.Linear(128, num_classes)
        
    def forward(self, x):
        x = self.conv1(x); x = self.conv2(x); x = self.conv3(x)
        x = self.drop(x); x = self.gap(x); x = x.view(x.size(0), -1)
        return self.fc(x)

def build_model(n_cls):
    m = Compact3DCNN(n_cls).to(DEVICE)
    print(f"Model: {sum(x.numel() for x in m.parameters()):,} params → {DEVICE}")
    return m

In [ ]:
import json, random, time, numpy as np, cv2
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter
from tqdm.auto import tqdm

try:
    import decord
    decord.bridge.set_bridge('numpy')
    USE_DECORD = True
    print("⚡ Using decord for fast video loading")
except ImportError:
    USE_DECORD = False
    print("📹 Using OpenCV for video loading (install decord for 3-5x speedup)")

EXTS = {".avi", ".mp4", ".mov", ".mkv", ".webm"}
SZ = IMG_SIZE[0]  # 224

class VideoDataset(Dataset):
    def __init__(self, paths, labels, is_train=True, use_cache=False, cache_dir=None):
        self.paths, self.labels, self.train = paths, labels, is_train
        self.use_cache = use_cache
        self.cache_dir = Path(cache_dir) if cache_dir else Path("/content/cache")
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        if self.use_cache:
            p = Path(self.paths[i])
            cpath = self.cache_dir / p.parent.name / f"{p.stem}.npy"
            if cpath.exists():
                try:
                    loaded = np.load(cpath)
                    if loaded.dtype == np.uint8:
                        frames = loaded.astype(np.float32) / 255.0
                    else:
                        frames = loaded.astype(np.float32)
                    return torch.from_numpy(frames).permute(3, 0, 1, 2), self.labels[i]
                except Exception:
                    pass
        try:
            frames = self._load_decord(self.paths[i]) if USE_DECORD else self._load_cv2(self.paths[i])
        except Exception:
            frames = np.zeros((N_FRAMES, SZ, SZ, 3), np.float32)
        return torch.from_numpy(frames).permute(3, 0, 1, 2).float(), self.labels[i]

    def _load_decord(self, path):
        """3-5x faster than OpenCV — batch frame access, no per-frame seeking."""
        vr = decord.VideoReader(str(path), num_threads=1)
        total = len(vr)
        need = 1 + (N_FRAMES - 1) * FRAME_STEP
        start = random.randint(0, max(0, total - need)) if self.train and total > need else 0
        idxs = [min(start + k * FRAME_STEP, total - 1) for k in range(N_FRAMES)]
        batch = vr.get_batch(idxs).asnumpy()  # (T, H, W, 3) uint8
        out = []
        for f in batch:
            f = f.astype(np.float32) / 255.0
            h, w = f.shape[:2]; s = min(SZ/h, SZ/w); nh, nw = int(h*s), int(w*s)
            f = cv2.resize(f, (nw, nh), cv2.INTER_LINEAR)
            ph, pw = SZ-nh, SZ-nw
            f = cv2.copyMakeBorder(f, ph//2, ph-ph//2, pw//2, pw-pw//2, cv2.BORDER_CONSTANT)
            out.append(f)
        return np.array(out, np.float32)

    def _load_cv2(self, path):
        cap = cv2.VideoCapture(str(path))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        need = 1 + (N_FRAMES - 1) * FRAME_STEP
        start = random.randint(0, max(0, total - need)) if self.train and total > need else 0
        
        frames = []
        if start > 0:
            cap.set(cv2.CAP_PROP_POS_FRAMES, start)
            
        target_indices = {k * FRAME_STEP for k in range(N_FRAMES)}
        max_idx = (N_FRAMES - 1) * FRAME_STEP
        
        curr_idx = 0
        while len(frames) < N_FRAMES:
            ok, f = cap.read()
            if not ok:
                break
            if curr_idx in target_indices:
                f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
                h, w = f.shape[:2]; s = min(SZ/h, SZ/w); nh, nw = int(h*s), int(w*s)
                f = cv2.resize(f, (nw, nh), cv2.INTER_LINEAR)
                ph, pw = SZ-nh, SZ-nw
                f = cv2.copyMakeBorder(f, ph//2, ph-ph//2, pw//2, pw-pw//2, cv2.BORDER_CONSTANT)
                frames.append(f)
            curr_idx += 1
            if curr_idx > max_idx:
                break
                
        while len(frames) < N_FRAMES:
            frames.append(np.zeros((SZ, SZ, 3), np.float32))
            
        cap.release()
        return np.array(frames, np.float32)

In [ ]:
# Generates .npy cache for 10x-50x faster training epochs.
# Bypasses slow I/O by pre-extracting/pre-copying via a consolidated tarball from Google Drive.
DRIVE_TAR = TEMP / "cache_uint8.tar"

if USE_CACHE:
    # A local cache is only valid if it exists and contains processed .npy files
    local_cache_valid = CACHE_DIR.exists() and any(CACHE_DIR.glob("**/*.npy"))
    
    if not local_cache_valid:
        if CACHE_DIR.exists():
            print(f"🧹 Clearing incomplete/empty local cache directory: {CACHE_DIR}")
            import shutil
            shutil.rmtree(CACHE_DIR, ignore_errors=True)
            
        # Try to locate the cache tarball on Google Drive (scanning multiple potential paths)
        import glob
        DRIVE_TAR_OPTIONS = [
            DRIVE_TAR,
            Path("/content/drive/MyDrive/cache_uint8.tar"),
            Path("/content/drive/MyDrive/Kinetics700-A/temp/cache_uint8.tar"),
            Path("/content/drive/MyDrive/Kinetics700/temp/cache_uint8.tar"),
            Path("/content/drive/MyDrive/Kinetics700-A/cache_uint8.tar"),
            Path("/content/drive/MyDrive/Kinetics700/cache_uint8.tar"),
            Path("/content/drive/MyDrive/Kinetics700-A_cache_uint8.tar"),
            Path("/content/drive/MyDrive/Kinetics700_cache_uint8.tar"),
        ]
        found_tar = None
        for opt in DRIVE_TAR_OPTIONS:
            try:
                if opt.exists():
                    found_tar = opt
                    break
            except Exception:
                pass
                
        # If not found at standard locations, search recursively or broadly in MyDrive
        if not found_tar:
            try:
                patterns = [
                    "/content/drive/MyDrive/*cache*.tar",
                    "/content/drive/MyDrive/Kinetics700-A/*cache*.tar",
                    "/content/drive/MyDrive/Kinetics700-A/temp/*cache*.tar",
                    "/content/drive/MyDrive/Kinetics700/*cache*.tar",
                    "/content/drive/MyDrive/Kinetics700/temp/*cache*.tar",
                ]
                for pattern in patterns:
                    matches = glob.glob(pattern)
                    if matches:
                        for match in matches:
                            if "cache" in Path(match).name.lower():
                                found_tar = Path(match)
                                break
                        if found_tar:
                            break
            except Exception:
                pass
                
        if found_tar:
            print(f"📦 Found consolidated cache tarball on Google Drive at: {found_tar}")
            print("Extracting cache directly to Colab local SSD (this takes ~1 minute)…")
            CACHE_DIR.mkdir(exist_ok=True, parents=True)
            import tarfile
            with tarfile.open(found_tar, "r") as tar:
                tar.extractall(path="/content/cache")
            print("✓ Consolidated cache successfully extracted to /content/cache!")
            local_cache_valid = True
            DRIVE_TAR = found_tar
            
    # 2. If no tarball exists, generate cache from scratch and pack it to Drive for future sessions
    if not (CACHE_DIR.exists() and any(CACHE_DIR.glob("**/*.npy"))):
        print("No preprocessed cache found. Generating fresh cache from raw videos…")
        CACHE_DIR.mkdir(exist_ok=True, parents=True)
        
        # Need index to know classes and files to cache
        print("📂 Scanning Kinetics-700 folders for caching...")
        classes_to_cache = sorted([d.name for d in DATASET_DIR.iterdir() if d.is_dir() and not d.name.startswith(".")])
        
        def extract_and_save(video_path, cls_name):
            out_dir = CACHE_DIR / cls_name
            out_dir.mkdir(exist_ok=True, parents=True)
            out_path = out_dir / f"{Path(video_path).stem}.npy"
            if out_path.exists(): return True
            try:
                ds = VideoDataset([str(video_path)], [0], is_train=False, use_cache=False)
                # Force cv2 for 100% stable, deadlock-free parallel processing in multiprocessing pools
                frames = ds._load_cv2(video_path)
                frames_uint8 = np.clip(frames * 255.0, 0, 255).astype(np.uint8)
                np.save(out_path, frames_uint8)
                return True
            except Exception:
                return False

        all_vids = []
        for c in classes_to_cache:
            vids = [str(f) for f in (DATASET_DIR / c).iterdir() if f.suffix.lower() in EXTS]
            if MAX_SAMPLES_PER_CLASS and len(vids) > MAX_SAMPLES_PER_CLASS:
                random.seed(SEED)
                vids = random.sample(vids, MAX_SAMPLES_PER_CLASS)
            for v in vids:
                all_vids.append((v, c))
                
        print(f"\nCaching {len(all_vids)} videos locally to {CACHE_DIR} using {CACHE_WORKERS} CPU workers...")
        from multiprocessing import Pool
        with Pool(CACHE_WORKERS) as p:
            r = list(tqdm(p.starmap(extract_and_save, all_vids), total=len(all_vids), desc="Generating .npy Cache"))
        print(f"✓ Cached {sum(r)}/{len(all_vids)} videos.")
        
        # Pack to tarball locally first (blazing fast and robust!)
        print("📦 Packaging consolidated cache to local SSD first (highly reliable)…")
        local_tar = Path("/content/cache_uint8.tar")
        import tarfile
        with tarfile.open(local_tar, "w") as tar:
            tar.add("/content/cache", arcname=".")
        print("✓ Local packaging complete!")
        
        # Copy to Google Drive
        print(f"📤 Copying consolidated cache tarball to Google Drive: {DRIVE_TAR}")
        print("This may take a minute. Please do not close the notebook until this completes.")
        TEMP.mkdir(parents=True, exist_ok=True)
        import shutil
        shutil.copy(local_tar, DRIVE_TAR)
        local_tar.unlink(missing_ok=True)
        print("✓ Tarball successfully copied to Google Drive!")
        
        # Force flush Google Drive to guarantee it's saved immediately
        print("⚡ Flushing Google Drive to guarantee files are synchronized…")
        try:
            from google.colab import drive
            drive.flush_and_unmount()
            print("✓ Google Drive flushed and unmounted successfully to secure all files!")
            print("ℹ️ Re-mounting Google Drive for training checkpoints…")
            drive.mount("/content/drive")
            print("✓ Google Drive successfully re-mounted!")
        except Exception as e:
            print(f"⚠️ Warning: Drive flush failed ({e}). Please keep Colab open for a minute to ensure background sync completes.")

In [ ]:
def build_loaders():
    # Try loading cached file index (avoids re-scanning 700 folders each restart)
    if INDEX_PATH.exists():
        idx = json.loads(INDEX_PATH.read_text())
        paths, labels, classes = idx["paths"], idx["labels"], idx["classes"]
        print(f"⚡ Loaded file index from cache ({len(paths)} samples, {len(classes)} classes)")
    else:
        print("📂 Scanning Kinetics-700 folders (one-time, saved for reuse)...")
        classes = sorted([d.name for d in DATASET_DIR.iterdir()
                          if d.is_dir() and not d.name.startswith(".")])
        c2i = {c: i for i, c in enumerate(classes)}
        paths, labels = [], []
        for c, i in tqdm(c2i.items(), desc="Indexing 700 classes"):
            vids = [str(f) for f in (DATASET_DIR / c).iterdir() if f.suffix.lower() in EXTS]
            if MAX_SAMPLES_PER_CLASS and len(vids) > MAX_SAMPLES_PER_CLASS:
                random.seed(SEED)
                vids = random.sample(vids, MAX_SAMPLES_PER_CLASS)
            for v in vids:
                paths.append(v); labels.append(i)
        INDEX_PATH.write_text(json.dumps({"paths": paths, "labels": labels, "classes": classes}))
        print(f"✓ Index saved → {INDEX_PATH}")

    print(f"Dataset: {len(paths)} samples | {len(classes)} classes")

    # Filter sparse classes
    cnt = Counter(labels)
    min_s = max(2, int(1 / VAL_SPLIT) + 1)
    ok = {l for l, n in cnt.items() if n >= min_s}
    if len(ok) < len(set(labels)):
        filt = [(p, l) for p, l in zip(paths, labels) if l in ok]
        paths = [p for p, _ in filt]; labels = [l for _, l in filt]
        o2n = {o: n for n, o in enumerate(sorted(ok))}
        labels = [o2n[l] for l in labels]
        classes = [classes[o] for o in sorted(ok)]

    tr_p, vl_p, tr_l, vl_l = train_test_split(
        paths, labels, test_size=VAL_SPLIT, random_state=SEED, stratify=labels)
    print(f"Split: {len(tr_p)} train / {len(vl_p)} val")

    use_cache = USE_CACHE and CACHE_DIR.exists() and any(CACHE_DIR.iterdir())
    cache_dir = CACHE_DIR if use_cache else None
    if use_cache:
        print(f"✓ Dataloader is using preprocessed cache from: {cache_dir}")
    else:
        print("⚠ Dataloader is using RAW videos (slower)")

    kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=HAS_GPU,
              prefetch_factor=PREFETCH, persistent_workers=True if NUM_WORKERS > 0 else False)
    tr = DataLoader(VideoDataset(tr_p, tr_l, is_train=True, use_cache=use_cache, cache_dir=cache_dir), shuffle=True, drop_last=True, **kw)
    vl = DataLoader(VideoDataset(vl_p, vl_l, is_train=False, use_cache=use_cache, cache_dir=cache_dir), shuffle=False, **kw)
    return tr, vl, classes

train_loader, val_loader, class_names = build_loaders()
NUM_CLASSES = len(class_names)
print(f"\n✓ {NUM_CLASSES} classes loaded")

In [ ]:
from torch.amp import GradScaler, autocast

BEST_CKPT   = CKPT_DIR / f"{EXPERIMENT}_best.pth"
LATEST_CKPT = CKPT_DIR / f"{EXPERIMENT}_latest.pth"
STATE_JSON  = CKPT_DIR / f"{EXPERIMENT}_state.json"

model     = build_model(NUM_CLASSES)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=ADAM_BETAS, eps=1e-8)
scaler    = GradScaler("cuda", enabled=USE_AMP)
criterion = nn.CrossEntropyLoss()

start_epoch  = 1
best_val_acc = 0.0
history = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[], "epoch_time":[], "lr":[]}

resume_ckpt = LATEST_CKPT if LATEST_CKPT.exists() else (BEST_CKPT if BEST_CKPT.exists() else None)
if resume_ckpt:
    ck = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model_state_dict"])
    optimizer.load_state_dict(ck["optimizer_state_dict"])
    best_val_acc = ck.get("best_val_acc", ck.get("val_accuracy", 0.0))
    start_epoch = ck["epoch"] + 1
    print(f"✓ Resumed from epoch {ck['epoch']} (best_acc={best_val_acc:.2%}) via {resume_ckpt.name}")

if STATE_JSON.exists():
    history = json.loads(STATE_JSON.read_text())
    keep = start_epoch - 1
    for k in history: history[k] = history[k][:keep]
    print(f"✓ History loaded & truncated to {keep} epochs")

In [ ]:
try:
    import graphviz
    dot = graphviz.Digraph(format='png', graph_attr={
        'rankdir': 'TB', 'bgcolor': 'white', 'dpi': '200',
        'pad': '0.5', 'nodesep': '0.4', 'ranksep': '0.5'
    })
    dot.attr('node', shape='record', style='filled,rounded', fontname='Helvetica', fontsize='11')
    dot.attr('edge', color='#555555', penwidth='1.5', arrowsize='0.8')
    dot.node('input',  f'Input\n(B, 3, {N_FRAMES}, 224, 224)', fillcolor='#dfe6e9')
    dot.node('conv1',  'Conv3D(3→32)+ReLU\nMaxPool3D(2)', fillcolor='#74b9ff')
    dot.node('conv2',  'Conv3D(32→64)+ReLU\nMaxPool3D(2)', fillcolor='#55efc4')
    dot.node('conv3',  'Conv3D(64→128)+ReLU\nMaxPool3D(2)', fillcolor='#ffeaa7')
    dot.node('drop',   f'Dropout({DROPOUT})', fillcolor='#fab1a0')
    dot.node('gap',    'GlobalAvgPool3D', fillcolor='#a29bfe')
    dot.node('fc',     f'Dense(128→{NUM_CLASSES})', fillcolor='#fd79a8')
    dot.node('out',    f'Output ({NUM_CLASSES})', fillcolor='#00b894')
    for a, b in [('input','conv1'),('conv1','conv2'),('conv2','conv3'),
                  ('conv3','drop'),('drop','gap'),('gap','fc'),('fc','out')]:
        dot.edge(a, b)
    p = str(FIG_DIR / f"{EXPERIMENT}_architecture")
    dot.render(p, cleanup=True)
    print(f"✓ Architecture → {p}.png")
    from IPython.display import Image, display
    display(Image(filename=p + ".png"))
except Exception as e:
    print(f"Arch viz skipped: {e}")

In [ ]:
from torch.utils.tensorboard import SummaryWriter
tb_writer = SummaryWriter(str(TB_DIR / EXPERIMENT))
print(f"✓ TensorBoard → {TB_DIR / EXPERIMENT}")

In [ ]:
def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss = correct = total = 0
    tag = "🏋️ Train" if training else "📊 Val"
    ctx = torch.enable_grad if training else torch.no_grad
    with ctx():
        if training:
            optimizer.zero_grad(set_to_none=True)
            
        for batch_idx, (vids, labels) in enumerate(loader):
            vids, labels = vids.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            
            with autocast("cuda", enabled=USE_AMP):
                out = model(vids)
                loss = criterion(out, labels)
                if training:
                    loss = loss / ACCUMULATION_STEPS
                    
            if training:
                scaler.scale(loss).backward()
                if (batch_idx + 1) % ACCUMULATION_STEPS == 0 or (batch_idx + 1) == len(loader):
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
            
            loss_val = (loss.item() * ACCUMULATION_STEPS) if training else loss.item()
            total_loss += loss_val * vids.size(0)
            correct += out.max(1)[1].eq(labels).sum().item()
            total += labels.size(0)
            
            # Print status update every 50 batches using carriage return, completely eliminating line spam!
            if batch_idx % 50 == 0 or (batch_idx + 1) == len(loader):
                print(f"\r    {tag} | Batch {batch_idx}/{len(loader)} | Loss: {total_loss/total:.4f} | Acc: {correct/total:.2%}", end="", flush=True)
                
        # Clean carriage return line when done
        print("\r" + " " * 90 + "\r", end="", flush=True)
            
    return total_loss / total, correct / total

print(f"\n{'='*65}")
print(f"  Training: {EXPERIMENT}")
print(f"  Epochs: {start_epoch} → {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LR}")
print(f"  Classes: {NUM_CLASSES} | AMP: {USE_AMP} | Device: {DEVICE}")
print(f"{'='*65}\n")

epoch_pbar = tqdm(range(start_epoch, EPOCHS + 1), desc="🔥 Training", unit="ep",
                  bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} ep '
                             '[{elapsed}<{remaining}, {rate_fmt}{postfix}]')

for epoch in epoch_pbar:
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, True)
    vl_loss, vl_acc = run_epoch(val_loader, False)
    elapsed = time.time() - t0

    history["train_loss"].append(tr_loss); history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc);   history["val_acc"].append(vl_acc)
    history["epoch_time"].append(elapsed); history["lr"].append(optimizer.param_groups[0]["lr"])

    tb_writer.add_scalars("Loss", {"train": tr_loss, "val": vl_loss}, epoch)
    tb_writer.add_scalars("Accuracy", {"train": tr_acc, "val": vl_acc}, epoch)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                     "optimizer_state_dict": optimizer.state_dict(),
                     "val_accuracy": vl_acc, "best_val_acc": best_val_acc,
                     "class_names": class_names,
                     "config": {"n_frames": N_FRAMES, "frame_step": FRAME_STEP,
                                "img_size": IMG_SIZE, "conv_filters": CONV_FILTERS,
                                "dropout": DROPOUT, "num_classes": NUM_CLASSES}},
                    str(BEST_CKPT))

    torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_accuracy": vl_acc, "best_val_acc": best_val_acc,
                "class_names": class_names}, str(LATEST_CKPT))
    STATE_JSON.write_text(json.dumps(history, indent=2))

    epoch_pbar.set_postfix_str(
        f"val_acc={vl_acc:.2%} | best={best_val_acc:.2%} | loss={vl_loss:.4f} | {elapsed:.0f}s/ep")

tb_writer.close()
print(f"\n✓ Training complete. Best val_acc: {best_val_acc:.2%}")

In [ ]:
import shutil
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report,
                              precision_recall_fscore_support, accuracy_score)

plt.rcParams.update({"font.size": 12, "font.family": "serif", "figure.dpi": 150,
                      "savefig.dpi": 300, "savefig.bbox": "tight",
                      "axes.grid": True, "grid.alpha": 0.3})

ck = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(ck["model_state_dict"])
model.eval()
print(f"✓ Loaded best checkpoint: epoch {ck['epoch']} | val_acc={ck['val_accuracy']:.2%}")

In [ ]:
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    eval_pbar = tqdm(val_loader, desc="📊 Evaluating", unit="batch")
    for vids, labs in eval_pbar:
        vids = vids.to(DEVICE, non_blocking=True)
        with autocast("cuda", enabled=USE_AMP):
            out = model(vids)
        probs = torch.softmax(out.float(), dim=1)
        preds_batch = probs.argmax(1).cpu().numpy()
        all_preds.extend(preds_batch)
        all_labels.extend(labs.numpy())
        all_probs.extend(probs.cpu().numpy())
        running_acc = np.mean(np.array(all_preds) == np.array(all_labels))
        eval_pbar.set_postfix(acc=f"{running_acc:.2%}")

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

In [ ]:
def compute_all_metrics(preds, labels, names):
    p, r, f, s = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)
    cm = confusion_matrix(labels, preds, labels=range(len(names)))
    spec = []
    for i in range(len(names)):
        tn = cm.sum() - cm[i].sum() - cm[:, i].sum() + cm[i, i]
        fp = cm[:, i].sum() - cm[i, i]
        spec.append(tn / (tn + fp) if (tn + fp) > 0 else 0.)
    acc = accuracy_score(labels, preds)
    mp, mr, mf, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": float(acc),
            "per_class": {names[i]: {"precision": float(p[i]), "recall": float(r[i]),
                "specificity": float(spec[i]), "f1": float(f[i]), "support": int(s[i])}
                for i in range(len(names))},
            "macro": {"precision": float(mp), "recall": float(mr),
                      "specificity": float(np.mean(spec)), "f1": float(mf)},
            "confusion_matrix": cm.tolist()}

metrics = compute_all_metrics(all_preds, all_labels, class_names)
print(f"\n{'='*50}")
print(f"  Overall Accuracy : {metrics['accuracy']:.2%}")
print(f"  Macro F1         : {metrics['macro']['f1']:.4f}")
print(f"  Macro Precision  : {metrics['macro']['precision']:.4f}")
print(f"  Macro Recall     : {metrics['macro']['recall']:.4f}")
print(f"{'='*50}")

metrics_path = METRICS_DIR / f"{EXPERIMENT}_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"✓ Metrics → {metrics_path}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(history["train_loss"]) + 1)
ax1.plot(ep, history["train_loss"], label="Train", lw=1.5, color="#e17055")
ax1.plot(ep, history["val_loss"], label="Val", lw=1.5, color="#0984e3")
ax1.fill_between(ep, history["train_loss"], alpha=0.1, color="#e17055")
ax1.fill_between(ep, history["val_loss"], alpha=0.1, color="#0984e3")
ax1.set(xlabel="Epoch", ylabel="Loss", title="Loss"); ax1.legend()
ax2.plot(ep, [v*100 for v in history["train_acc"]], label="Train", lw=1.5, color="#e17055")
ax2.plot(ep, [v*100 for v in history["val_acc"]], label="Val", lw=1.5, color="#0984e3")
ax2.set(xlabel="Epoch", ylabel="Accuracy %", title="Accuracy"); ax2.legend()
plt.tight_layout(); fig.savefig(FIG_DIR / f"{EXPERIMENT}_curves.png"); plt.show(); plt.close()

In [ ]:
cm = np.array(metrics["confusion_matrix"])
n = len(class_names)
show_n = min(30, n)
if n > 30:
    # Show top-30 by support
    supports = [metrics["per_class"][c]["support"] for c in class_names]
    top_idx = sorted(range(n), key=lambda i: supports[i], reverse=True)[:30]
    show_names = [class_names[i] for i in top_idx]
    show_cm = cm[np.ix_(top_idx, top_idx)]
    title_suffix = " (Top-30 by support)"
else:
    show_names = class_names; show_cm = cm; title_suffix = ""

for norm, suffix, title in [(False, "_cm.png", f"Confusion Matrix{title_suffix}"),
                             (True, "_cm_norm.png", f"CM Normalised{title_suffix}")]:
    d = show_cm.astype(float) / show_cm.sum(axis=1, keepdims=True).clip(1) if norm else show_cm
    fig, ax = plt.subplots(figsize=(12, 12))
    sns.heatmap(d, annot=show_n<=30, fmt=".2f" if norm else "d", cmap="Blues",
                xticklabels=show_names, yticklabels=show_names, ax=ax, square=True, linewidths=0.3)
    ax.set(xlabel="Predicted", ylabel="True", title=title)
    plt.xticks(rotation=90, fontsize=6); plt.yticks(fontsize=6)
    plt.tight_layout(); fig.savefig(FIG_DIR / f"{EXPERIMENT}{suffix}"); plt.show(); plt.close()

In [ ]:
f1s = {c: metrics["per_class"][c]["f1"] for c in class_names}
srt = sorted(f1s.items(), key=lambda x: x[1])
bot5 = srt[:5]; top5 = srt[-5:][::-1]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, ttl, col in [(axes[0], bot5, "Bottom-5 F1", "#e17055"),
                             (axes[1], top5, "Top-5 F1", "#00b894")]:
    cs = [x[0][:20] for x in data]; vs = [x[1] for x in data]
    ax.barh(cs, vs, color=col, alpha=0.85); ax.set(xlim=(0, 1), title=ttl)
plt.tight_layout(); fig.savefig(FIG_DIR / f"{EXPERIMENT}_top_bottom5.png"); plt.show(); plt.close()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(history["epoch_time"]) + 1), history["epoch_time"], lw=1.5, color="#6c5ce7")
avg_t = np.mean(history["epoch_time"])
ax.axhline(avg_t, ls="--", color="#e17055", label=f"Avg: {avg_t:.1f}s")
ax.set(xlabel="Epoch", ylabel="Seconds", title="Epoch Duration"); ax.legend()
plt.tight_layout(); fig.savefig(FIG_DIR / f"{EXPERIMENT}_speed.png"); plt.show(); plt.close()

In [ ]:
import imageio

class GradCAM3D:
    def __init__(self, model):
        self.model = model; self.model.eval()
        self.act = self.grad = None
        model.conv3[0].register_forward_hook(lambda m, i, o: setattr(self, "act", o.detach()))
        model.conv3[0].register_full_backward_hook(lambda m, gi, go: setattr(self, "grad", go[0].detach()))
    def __call__(self, x, cls=None):
        x.requires_grad_(True); self.model.zero_grad()
        with torch.enable_grad():
            out = self.model(x)
            if cls is None: cls = out.argmax(1).item()
            out[0, cls].backward()
        w = self.grad.mean(dim=(2, 3, 4), keepdim=True)
        cam = F.relu((w * self.act).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=(N_FRAMES, *IMG_SIZE), mode="trilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        mn, mx = cam.min(), cam.max()
        return (cam - mn) / (mx - mn + 1e-8), cls

gcam = GradCAM3D(model)
for cls in tqdm(class_names[:5], desc="Grad-CAM"):
    if not (DATASET_DIR / cls).exists(): continue
    vids = [f for f in (DATASET_DIR / cls).iterdir() if f.suffix.lower() in EXTS]
    if not vids: continue
    ds = VideoDataset([str(vids[0])], [0], is_train=False)
    frames, _ = ds[0]
    inp = frames.unsqueeze(0).to(DEVICE)
    hm, pred = gcam(inp.clone())
    fr = frames.permute(1, 2, 3, 0).numpy()
    gif_frames = []
    for t in range(N_FRAMES):
        f = (fr[t] * 255).clip(0, 255).astype(np.uint8)
        h = (hm[t] * 255).astype(np.uint8)
        hc = cv2.cvtColor(cv2.applyColorMap(h, cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
        gif_frames.append(cv2.addWeighted(f, 0.6, hc, 0.4, 0))
    out_path = GRADCAM_DIR / f"{cls}_gradcam.gif"
    imageio.mimsave(str(out_path), gif_frames, duration=120, loop=0)
    print(f"  {cls} → predicted: {class_names[pred]}")
    from IPython.display import Image as IPyImage, display
    display(IPyImage(filename=str(out_path)))
print(f"✓ Grad-CAM GIFs → {GRADCAM_DIR}")

In [ ]:
final_save = {
    "model_state_dict": model.state_dict(),
    "class_names": class_names, "num_classes": NUM_CLASSES,
    "best_val_accuracy": best_val_acc,
    "config": {"architecture": "Compact3DCNN", "conv_filters": CONV_FILTERS,
               "dropout": DROPOUT, "n_frames": N_FRAMES, "frame_step": FRAME_STEP,
               "img_size": IMG_SIZE, "learning_rate": LR, "batch_size": BATCH_SIZE,
               "epochs_trained": len(history["train_loss"])},
    "metrics": {"accuracy": metrics["accuracy"], "macro_f1": metrics["macro"]["f1"],
                "macro_precision": metrics["macro"]["precision"],
                "macro_recall": metrics["macro"]["recall"]}
}
save_path = SAVED_DIR / f"{EXPERIMENT}_final.pth"
torch.save(final_save, str(save_path))
print(f"\n✓ Final model → {save_path}")

In [ ]:
report = {
    "experiment": EXPERIMENT, "dataset": "Kinetics-700",
    "architecture": "Compact3DCNN", "device": DEVICE,
    "total_params": sum(p.numel() for p in model.parameters()),
    "hyperparameters": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR,
        "n_frames": N_FRAMES, "frame_step": FRAME_STEP, "dropout": DROPOUT,
        "val_split": VAL_SPLIT, "mixed_precision": USE_AMP},
    "results": metrics,
    "training_summary": {
        "total_epochs": len(history["train_loss"]),
        "best_val_accuracy": best_val_acc,
        "avg_epoch_time_sec": float(np.mean(history["epoch_time"])) if history["epoch_time"] else None}
}
report_path = METRICS_DIR / f"{EXPERIMENT}_full_report.json"
report_path.write_text(json.dumps(report, indent=2))
print(f"✓ Report → {report_path}")

In [ ]:
print(f"\n{'='*65}")
print(f"  🎉 PIPELINE COMPLETE — {EXPERIMENT}")
print(f"{'='*65}")
print(f"  Model    : Compact3DCNN ({sum(p.numel() for p in model.parameters()):,} params)")
print(f"  Accuracy : {metrics['accuracy']:.2%}")
print(f"  Macro F1 : {metrics['macro']['f1']:.4f}")
print(f"  Best Acc : {best_val_acc:.2%}")
print(f"  Saved to : {SAVED_DIR}")
print(f"{'='*65}")
print(f"\n📂 All outputs in: {TEMP}")